In [2]:
import vtk
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.spatial import KDTree
from scipy.interpolate import RegularGridInterpolator
from scipy.signal import savgol_filter
import pygmt
import importlib
import pandas as pd
import h5py
from scipy.special import erf
import matplotlib.colors as mcolors
from scipy.interpolate import interp1d
import matplotlib.patches as patches

os.chdir("../")
import parallel_curves
os.chdir("scripts_for_figures")

In [10]:
panel_width  = 12.0
panel_height = 8.0

panel_dim = [0, 0, panel_width, panel_height]

fig = plt.figure(dpi=100, figsize=(panel_width, panel_height))
main_ax = plt.axes(panel_dim)

for spine in main_ax.spines.values():
    spine.set_linewidth(15)

# The errors were calculated using the ABSOLUTE LOG MISFIT, and copied
# manually into the arrays below from the other python script `comparison_to_NAIF.ipynb.
# The indices of the error arrays are:
# 0: No faults, no flexure
# 1: Faults (25x permeability), no flexure
# 2: No faults, with flexure
# 3: Faults (2.5x permeability), with flexure
# 4: Faults (5x permeability), with flexure
# 5: Faults (7.5x permeability), with flexure
# 6: Faults (10x permeability), with flexure
# 7: Faults (25x permeability), with flexure

hatakeyama_low_errors  = np.array([0.61651971, 0.62490163, 0.46321065, 0.46325271,\
                                   0.46597517, 0.47550359, 0.50052289, 0.54337957])
hatakeyama_high_errors = np.array([0.65348662, 0.65600316, 0.60061538, 0.60167441, \
                                   0.60674994, 0.61116935, 0.61544828, 0.61869375])
kuang_jiao_errors      = np.array([0.66089194, 0.66065577, 0.6201892, 0.62021343, \
                                   0.620529, 0.62075678, 0.62094497, 0.621077])
power_law_errors       = np.array([0.66070083, 0.66064111, 0.62057284, 0.62057563, \
                                   0.62080093, 0.6209431 , 0.62104376, 0.62110363])

model_type_x      = np.arange(1, 9, 1)
kuang_jiao_y      = np.zeros(len(model_type_x)) + 1
power_law_y       = np.zeros(len(model_type_x)) + 2
hatakeyama_high_y = np.zeros(len(model_type_x)) + 3
hatakeyama_low_y  = np.zeros(len(model_type_x)) + 4

rect = patches.Rectangle((0.5, 0.5), 2, 5, linewidth=1, edgecolor='black', facecolor='black', alpha=0.1)
main_ax.add_patch(rect)
main_ax.axvline(2.5, lw=30, c='k')

levels = np.linspace(0.46, 0.66, 21)
norm = mcolors.BoundaryNorm(levels, ncolors=plt.cm.viridis.N, clip=True)

cax = fig.add_axes([0, -3, panel_width, 0.5])

c_map = "viridis_r"

im = main_ax.scatter(model_type_x, kuang_jiao_y, c=kuang_jiao_errors, 
                     s=200000, cmap=c_map, edgecolor="black", norm=norm, 
                     marker='o', lw=10, zorder=100000)

cbar = fig.colorbar(im, cax=cax, orientation='horizontal', norm=norm)
cbar.set_label('$\chi$ Misfit', size=400)
cbar.set_ticks([0.46, 0.51, 0.56, 0.61, 0.66])
cbar.ax.tick_params(which="major", labelsize=250, length=200, width=25)
cbar.ax.tick_params(which="minor", labelsize=0, length=75, width=10) 

main_ax.scatter(model_type_x, power_law_y, c=power_law_errors, 
                     s=200000, cmap=c_map, edgecolor="black", norm=norm, 
                     marker='o', lw=10, zorder=100000)
main_ax.scatter(model_type_x, hatakeyama_high_y, c=hatakeyama_high_errors, 
                     s=200000, cmap=c_map, edgecolor="black", norm=norm, 
                     marker='o', lw=10, zorder=100000)
main_ax.scatter(model_type_x, hatakeyama_low_y, c=hatakeyama_low_errors, 
                     s=200000, cmap=c_map, edgecolor="black", norm=norm, 
                     marker='o', lw=10, zorder=100000)

main_ax.set_xlim(0.75, 8.25)
main_ax.set_ylim(0.75, 4.25)

main_ax.tick_params(axis="both", length=200, width=25)
main_ax.set_yticks([1, 2, 3, 4])

main_ax.text(0.4, 4.75, "a)", fontsize=300)

main_ax.text(8.8, 1.45, "Higher Background", fontsize=300, rotation=-90)
main_ax.text(8.5, 1.75, "Permeability", fontsize=300, rotation=-90)
main_ax.arrow(x=8.45, y=4.25, dx=0.0, dy=-3, width=0.1, clip_on=False, color='black')

main_ax.text(4.7, 4.85, "Higher Fault", fontsize=300)
main_ax.text(4.7, 4.55, "Permeability", fontsize=300)
main_ax.arrow(x=3, y=4.45, dx=4.75, dy=0, width=0.1, clip_on=False, color='black')

main_ax.text(1.0, 4.5, "No Flexure", fontsize=300)

main_ax.text(0, 0.6, "Reference Permeability Model", fontsize=300, rotation=90, fontweight="bold")

main_ax.text(0.4, 0.65, "Model 1", fontsize=235, rotation=90)
main_ax.text(0.4, 1.65, "Model 2", fontsize=235, rotation=90)
main_ax.text(0.4, 2.65, "Model 3", fontsize=235, rotation=90)
main_ax.text(0.4, 3.65, "Model 4", fontsize=235, rotation=90)

main_ax.text(0.875, 0.4, "1x",   fontsize=235, rotation=0)
main_ax.text(1.875, 0.4, "25x",  fontsize=235, rotation=0)
main_ax.text(2.875, 0.4, "1x",   fontsize=235, rotation=0)
main_ax.text(3.875, 0.4, "2.5x", fontsize=235, rotation=0)
main_ax.text(4.875, 0.4, "5x",   fontsize=235, rotation=0)
main_ax.text(5.875, 0.4, "10x", fontsize=235, rotation=0)
main_ax.text(6.875, 0.4, "25x",  fontsize=235, rotation=0)
main_ax.text(7.865, 0.4, "100x",  fontsize=235, rotation=0)

main_ax.text(2.5, 0., "Fault Permeability Enhancement", fontsize=300, rotation=0, fontweight="bold")

plt.savefig("Figures/error_plot.png", bbox_inches="tight")
plt.show()